In [1]:
from finn.util.pytorch import ToTensor
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType
import torch
import onnx
from finn.util.test import get_test_model_trained
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs

In [2]:
import torch
import onnx
from finn.util.test import get_test_model_trained
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs

build_dir = "."

export_onnx_path = build_dir + "/finn_ready_model.onnx"
qonnx_cleanup(export_onnx_path, out_file=export_onnx_path)
model = ModelWrapper(export_onnx_path)
model = model.transform(ConvertQONNXtoFINN())
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(RemoveStaticGraphInputs())
model.save(build_dir + "/finn_ready_model_tidy.onnx")

In [3]:
build_dir = "."
model = ModelWrapper(build_dir+"/finn_ready_model_tidy.onnx")

global_inp_name = model.graph.input[0].name
ishape = model.get_tensor_shape(global_inp_name)
# preprocessing: torchvision's ToTensor divides uint8 inputs by 255
totensor_pyt = ToTensor()
# chkpt_preproc_name = "./2dconv_model_preproc.onnx"
# export_qonnx(totensor_pyt, torch.randn(ishape), chkpt_preproc_name)
# qonnx_cleanup(chkpt_preproc_name, out_file=chkpt_preproc_name)
# pre_model = ModelWrapper(chkpt_preproc_name)
# pre_model = pre_model.transform(ConvertQONNXtoFINN())

# join preprocessing and core model
# model = model.transform(MergeONNXModels(pre_model))
# add input quantization annotation: UINT8 for all BNN-PYNQ models
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

In [4]:
from qonnx.transformation.insert_topk import InsertTopK
from qonnx.transformation.infer_datatypes import InferDataTypes

# postprocessing: insert Top-1 node at the end
# model = model.transform(InsertTopK(k=1))
chkpt_name = "./2dconv_model_pre.onnx"
# tidy-up again
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())
model.save(chkpt_name)

In [5]:
from finn.transformation.streamline import Streamline
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.bipolar_to_xnor import ConvertBipolarMatMulToXnorPopcount
import finn.transformation.streamline.absorb as absorb
from finn.transformation.streamline.reorder import MakeMaxPoolNHWC, MoveScalarLinearPastInvariants
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors

model = ModelWrapper(build_dir + "/2dconv_model_pre.onnx")
model = model.transform(MoveScalarLinearPastInvariants())
model = model.transform(Streamline())
model = model.transform(LowerConvsToMatMul())
model = model.transform(MakeMaxPoolNHWC())
model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())
# model = model.transform(ConvertBipolarMatMulToXnorPopcount())
model = model.transform(Streamline())
# absorb final add-mul nodes into TopK
# model = model.transform(absorb.AbsorbScalarMulAddIntoTopK())
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())
model.save(build_dir + "/2dconv_model_streamlined.onnx")

/mnt/c/Users/An/finn/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:127: UserWarning: Assuming 4D input is NCHW
  warnings.warn("Assuming 4D input is NCHW")


In [6]:
from finn.util.basic import pynq_part_map
# change this if you have a different PYNQ board, see list above
fpga_part = "xc7z010clg400-1"

import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.create_dataflow_partition import (
    CreateDataflowPartition,
)
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from finn.transformation.streamline.reorder import MakeMaxPoolNHWC, MoveScalarLinearPastInvariants


model = ModelWrapper(build_dir + "/2dconv_model_streamlined.onnx")
model = model.transform(to_hw.InferBinaryMatrixVectorActivation())
model = model.transform(to_hw.InferQuantizedMatrixVectorActivation())
# TopK to LabelSelect
# model = model.transform(to_hw.InferLabelSelectLayer())
# input quantization (if any) to standalone thresholding
model = model.transform(to_hw.InferThresholdingLayer())
model = model.transform(to_hw.InferConvInpGen())
model = model.transform(MakeMaxPoolNHWC())
model = model.transform(to_hw.InferStreamingMaxPool())
# get rid of Reshape(-1, 1) operation between hw nodes
model = model.transform(RemoveCNVtoFCFlatten())
# get rid of Tranpose -> Tranpose identity seq
model = model.transform(absorb.AbsorbConsecutiveTransposes())
# infer tensor data layouts
model = model.transform(InferDataLayouts())
parent_model = model.transform(CreateDataflowPartition())
parent_model.save(build_dir + "/2dconv_model_dataflow_parent.onnx")
sdp_node = parent_model.get_nodes_by_op_type("StreamingDataflowPartition")[0]
sdp_node = getCustomOp(sdp_node)
dataflow_model_filename = sdp_node.get_nodeattr("model")
# save the dataflow partition with a different name for easier access
# and specialize the layers to HLS variants
dataflow_model = ModelWrapper(dataflow_model_filename)
# dataflow_model = dataflow_model.transform(SpecializeLayers(fpga_part))
dataflow_model.save(build_dir + "/2dconv_model_dataflow_model.onnx")

In [7]:
from finn.util.visualization import showInNetron
import os

# build_dir = os.environ["FINN_BUILD_DIR"]
# showInNetron("./finn_ready_model.onnx")
showInNetron("output/intermediate_models/step_create_stitched_ip.onnx")
# showInNetron("./2dconv_model_dataflow_model.onnx")
# showInNetron("./2dconv_model_pre.onnx")
# showInNetron("./2dconv_model_dataflow_parent.onnx")
# showInNetron("./2dconv_model_streamlined.onnx")


Serving 'output/intermediate_models/step_create_stitched_ip.onnx' at http://0.0.0.0:8081


In [8]:
# from functools import partial
# from finn.analysis.fpgadataflow.exp_cycles_per_layer import exp_cycles_per_layer
# from finn.analysis.fpgadataflow.res_estimation import res_estimation

# from qonnx.core.modelwrapper import ModelWrapper
# from qonnx.transformation.general import GiveUniqueNodeNames

# model = ModelWrapper(build_dir + "/2dconv_model_dataflow_model.onnx")
# model = model.transform(GiveUniqueNodeNames())

# cycles_dict = model.analysis(exp_cycles_per_layer)
# cycles_dict

In [9]:
# from qonnx.custom_op.registry import getCustomOp
# # Inspection script
# model = ModelWrapper(build_dir + "/2dconv_model_dataflow_model.onnx")
# fc_layers = model.get_nodes_by_op_type("MVAU_hls")
# # fc_layers
# # Iterate and print MW / MH
# for n in fc_layers:
#     inst = getCustomOp(n, model)
    
#     print(f"  MW: {inst.get_nodeattr('MW')}")
#     print(f"  MH: {inst.get_nodeattr('MH')}")


In [10]:
# model = ModelWrapper(build_dir + "/2dconv_model_dataflow_model.onnx")
# fc_layers = model.get_nodes_by_op_type("MVAU_hls")
# # each tuple is (PE, SIMD, in_fifo_depth) for a layer
# folding = [
#     (4, 2, [128]),
#     (8, 4, [128]),
# ]
# for fcl, (pe, simd, ififodepth) in zip(fc_layers, folding):
#     fcl_inst = getCustomOp(fcl)
#     fcl_inst.set_nodeattr("PE", pe)
#     fcl_inst.set_nodeattr("SIMD", simd)
#     fcl_inst.set_nodeattr("inFIFODepths", ififodepth)

# # # use same SIMD values for the sliding window operators
# # swg_layers = model.get_nodes_by_op_type("ConvolutionInputGenerator_rtl")
# # for i in range(len(swg_layers)):
# #     swg_inst = getCustomOp(swg_layers[i])
# #     simd = folding[i][1]
# #     swg_inst.set_nodeattr("SIMD", simd)

# model = model.transform(GiveUniqueNodeNames())
# model.save(build_dir + "/2dconv_model_folded.onnx")

In [11]:
# from functools import partial
# from finn.analysis.fpgadataflow.exp_cycles_per_layer import exp_cycles_per_layer
# from finn.analysis.fpgadataflow.res_estimation import res_estimation

# from qonnx.core.modelwrapper import ModelWrapper
# from qonnx.transformation.general import GiveUniqueNodeNames

# model = ModelWrapper(build_dir + "/2dconv_model_folded.onnx")
# model = model.transform(GiveUniqueNodeNames())

# cycles_dict = model.analysis(exp_cycles_per_layer)
# cycles_dict

In [12]:
# model_dwc = ModelWrapper("2dconv_model_folded.onnx")
# res_dict_dwc = model_dwc.analysis(partial(res_estimation, fpgapart="xc7z020clg400-1"))
# res_dict_dwc

In [13]:
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
import os
import shutil

## Build flow with hardware build
build_dir = "."
# model_file = build_dir+"/2dconv_model_folded.onnx"
model_file = build_dir+"/2dconv_model_dataflow_model.onnx"

output_dir = build_dir + "/output"

#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")

build_steps = [
    "step_specialize_layers",
    "step_target_fps_parallelization",
    "step_minimize_bit_width",
    "step_generate_estimate_reports",
    "step_hw_codegen",
    "step_hw_ipgen",
    "step_set_fifo_depths",
    "step_create_stitched_ip",
    "step_measure_rtlsim_performance",
    "step_out_of_context_synthesis",
    # "step_synthesize_bitfile",
    # "step_make_pynq_driver",
]

cfg_build = build.DataflowBuildConfig(
    output_dir                    = output_dir,
    mvau_wwidth_max     = 80,
    target_fps          = 200000,
    synth_clk_period_ns = 10.0,
    #specialize_layers_config_file = "specialize_layers_all_hls.json",
    # folding_config_file           = "cnv-w2a2_folding_config.json",
    fpga_part                         = "xc7z010clg400-1",
    shell_flow_type               = build_cfg.ShellFlowType.VIVADO_ZYNQ,
    folding_config_file           = "final_hw_config.json",
    # steps                         = build_steps,
    default_swg_exception         = True,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
        build_cfg.DataflowOutputType.STITCHED_IP,
        build_cfg.DataflowOutputType.RTLSIM_PERFORMANCE,
        build_cfg.DataflowOutputType.OOC_SYNTH,
        # build_cfg.DataflowOutputType.BITFILE,
        # build_cfg.DataflowOutputType.PYNQ_DRIVER,
        # build_cfg.DataflowOutputType.DEPLOYMENT_PACKAGE,
    ],
)



Previous run results deleted!


In [14]:
import os
# os.chdir('/mnt/c/Users/An/finn/notebooks/sin_pulse')
build.build_dataflow_cfg(model_file, cfg_build)

Running step: step_qonnx_to_finn [1/19]
Running step: step_tidy_up [2/19]
Running step: step_streamline [3/19]


Building dataflow accelerator from ./2dconv_model_dataflow_model.onnx
Intermediate outputs will be generated in /tmp/finn_dev_cutesquare
Final outputs will be generated in ./output
Build log is at ./output/build_dataflow.log
Running step: step_qonnx_to_finn [1/19]
Running step: step_tidy_up [2/19]
Running step: step_streamline [3/19]


Running step: step_convert_to_hw [4/19]
Running step: step_create_dataflow_partition [5/19]
Running step: step_specialize_layers [6/19]
Running step: step_target_fps_parallelization [7/19]
/mnt/c/Users/An/finn/src/finn/transformation/fpgadataflow/set_folding.py:221: UserWarning: SetFolding doesn't know how to handle op_type StreamingMaxPool_hls
  warnings.warn("SetFolding doesn't know how to handle op_type " + op_type)
/mnt/c/Users/An/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for layer StreamingMaxPool_hls_0 can be lower than
             actual latency!
  warnings.warn(
Running step: step_apply_folding_config [8/19]
/mnt/c/Users/An/finn/deps/qonnx/src/qonnx/transformation/general.py:354: UserWarning:
No HW configuration for nodes: Thresholding_rtl_0
  warnings.warn("\nNo HW configuration for nodes: " + ", ".join(missing_configurations))
/mnt/c/Users/An/finn/deps/qonnx/src/qonnx/transformation/general.py:358: UserWarning:
Unused HW con

Running step: step_convert_to_hw [4/19]
Running step: step_create_dataflow_partition [5/19]
Running step: step_specialize_layers [6/19]
Running step: step_target_fps_parallelization [7/19]
Running step: step_apply_folding_config [8/19]
Running step: step_minimize_bit_width [9/19]


Running step: step_generate_estimate_reports [10/19]
Running step: step_hw_codegen [11/19]


...

%Warning-MODDUP: /tmp/finn_dev_cutesquare/vivado_stitch_proj_nylprhtd/finn_design_wrapper.v:4491:8: Duplicate declaration of module: 'dwc_axi'
 4491 | module dwc_axi #(
      |        ^~~~~~~
                 /tmp/finn_dev_cutesquare/vivado_stitch_proj_nylprhtd/finn_design_wrapper.v:1993:8: ... Location of original declaration
 1993 | module dwc_axi #(
      |        ^~~~~~~
                 ... For warning description see https://verilator.org/warn/MODDUP?v=4.224
                 ... Use "/* verilator lint_off MODDUP */" and lint_on around source to disable this message.
%Warning-MODDUP: /tmp/finn_dev_cutesquare/vivado_stitch_proj_nylprhtd/finn_design_wrapper.v:8665:8: Duplicate declaration of module: 'dwc_axi'
 8665 | module dwc_axi #(
      |        ^~~~~~~
                 /tmp/finn_dev_cutesquare/vivado_stitch_proj_nylprhtd/finn_design_wrapper.v:1993:8: ... Location of original declaration
 1993 | module dwc_axi #(
      |        ^~~~~~~
%Warning-MODDUP: /tmp/finn_dev_cutesquare/v

Running step: step_out_of_context_synthesis [16/19]


ap_clk
xc7z010clg400-1
10.000000
0

****** Vivado v2024.2 (64-bit)
  **** SW Build 5239630 on Fri Nov 08 22:34:34 MST 2024
  **** IP Build 5239520 on Sun Nov 10 16:12:51 MST 2024
  **** SharedData Build 5239561 on Fri Nov 08 14:39:27 MST 2024
  **** Start of session at: Thu Dec 18 16:00:04 2025
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2024 Advanced Micro Devices, Inc. All Rights Reserved.

source finn_design_wrapper.tcl
# set fpga_part "xc7z010clg400-1"
# set origin_dir "."
# if { [info exists ::origin_dir_loc] } {
#   set origin_dir $::origin_dir_loc
# }
# variable script_file
# set script_file "vivadocompile.tcl"
# proc help {} {
#   variable script_file
#   puts "\nDescription:"
#   puts "Recreate a Vivado project from this script. The created project will be"
#   puts "functionally equivalent to the original project for which this script was"
#   puts "generated. The script contains commands for creating a project, filesets,"
#   puts "runs

Running step: step_synthesize_bitfile [17/19]
Running step: step_make_pynq_driver [18/19]
Running step: step_deployment_package [19/19]
Completed successfully


0